##Conexão com Google Drive

O Google Drive foi integrado ao ambiente do Google Colab para permitir o acesso ao conjunto de dados.

A conexão foi realizada em modo somente leitura (readonly=True), garantindo a integridade dos arquivos durante o treinamento.

Em seguida, foi definido o caminho base do dataset (data_path), facilitando o acesso às imagens e aos arquivos de anotação utilizados pelo modelo YOLO.

In [35]:
from google.colab import drive
import os

drive.mount('/content/drive', readonly=True)
data_path = '/content/drive/MyDrive/fase_6'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Instalação da biblioteca YOLO

A biblioteca Ultralytics foi instalada no ambiente Google Colab utilizando o gerenciador de pacotes pip. Essa biblioteca fornece a implementação do modelo YOLO, permitindo a realização de tarefas de detecção de objetos com suporte a aceleração por GPU.

In [5]:
!pip install ultralytics

##Importação da biblioteca YOLO

A biblioteca Ultralytics foi importada no ambiente para disponibilizar o modelo YOLO, permitindo a execução das etapas de treinamento, validação e inferência do modelo de detecção de objetos.

In [6]:
from ultralytics import YOLO

print("YOLO carregado com sucesso")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO carregado com sucesso


##Verificação da estrutura de arquivos

Antes do início do treinamento, foi realizada a verificação da estrutura do dataset utilizando a biblioteca os.

Esse procedimento permite confirmar se as pastas e arquivos (imagens e rótulos) estão organizados corretamente, evitando erros durante o treinamento do modelo YOLO.

In [34]:
import os

base_path = "/content/drive/MyDrive/Fase_6"

for root, dirs, files in os.walk(base_path):
    print(root, len(files))

/content/drive/MyDrive/Fase_6 1
/content/drive/MyDrive/Fase_6/images 0
/content/drive/MyDrive/Fase_6/images/test 8
/content/drive/MyDrive/Fase_6/images/train 64
/content/drive/MyDrive/Fase_6/images/val 8
/content/drive/MyDrive/Fase_6/labels 0
/content/drive/MyDrive/Fase_6/labels/test 8
/content/drive/MyDrive/Fase_6/labels/train 64
/content/drive/MyDrive/Fase_6/labels/val 8


## Carregamento do modelo YOLO

O modelo YOLOv8 na versão pré-treinada (yolov8n.pt) foi carregado para iniciar o processo de treinamento.

A escolha da versão nano (yolov8n) se deve ao seu baixo custo computacional e rapidez de treinamento, sendo adequada para conjuntos de dados pequenos e para execução em ambientes como o Google Colab.

Além disso, o uso de pesos pré-treinados permite aproveitar conhecimentos prévios do modelo, acelerando o aprendizado e melhorando o desempenho na tarefa de detecção de objetos.

In [23]:
model = YOLO("yolov8n.pt")

##Treinamento do modelo (30 épocas)

O modelo foi treinado utilizando o dataset configurado no arquivo dataset.yaml, por um total de 30 épocas.

Foram definidos parâmetros como o tamanho das imagens (imgsz=640) e o tamanho do lote (batch=8), buscando um equilíbrio entre qualidade das detecções e uso de memória da GPU no ambiente do Google Colab.

O número de épocas foi escolhido para permitir que o modelo aprendesse os padrões do dataset sem prolongar excessivamente o tempo de treinamento, considerando o tamanho reduzido da base de dados.

Esse processo possibilitou ao modelo aprender a identificar e localizar os objetos de interesse (mouse e apontador) nas imagens.

In [24]:
results_30 = model.train(
    data="/content/drive/MyDrive/Fase_6/dataset.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    name="exp_30_epochs"
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Fase_6/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp_30_epochs-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

## Inferência com o modelo treinado

Após o treinamento, o modelo foi carregado a partir dos melhores pesos gerados (best.pt) e utilizado para realizar predições em imagens do conjunto de teste.

Essa etapa, conhecida como inferência, permite avaliar na prática a capacidade do modelo em identificar e localizar os objetos (mouse e apontador) em novas imagens não utilizadas no treinamento.

In [25]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/exp_30_epochs-2/weights/best.pt")

model.predict(
    source="/content/drive/MyDrive/Fase_6/images/test",
    save=True
)


image 1/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_33.jpg: 640x480 2 apontadors, 63.5ms
image 2/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_34.jpg: 640x480 1 apontador, 6.6ms
image 3/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_35.jpg: 640x480 1 apontador, 7.2ms
image 4/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_36.jpg: 640x480 1 apontador, 6.5ms
image 5/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_33.jpg: 640x480 2 mouses, 6.4ms
image 6/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_34.jpg: 640x480 1 mouse, 9.4ms
image 7/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_35.jpg: 640x480 1 mouse, 7.4ms
image 8/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_36.jpg: 640x480 1 mouse, 6.8ms
Speed: 4.0ms preprocess, 14.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/detect/predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'mouse', 1: 'apontador'}
 obb: None
 orig_img: array([[[167, 183, 189],
         [167, 183, 189],
         [169, 185, 191],
         ...,
         [138, 150, 156],
         [140, 150, 157],
         [141, 151, 158]],
 
        [[167, 183, 189],
         [167, 183, 189],
         [168, 184, 190],
         ...,
         [141, 153, 159],
         [141, 151, 158],
         [141, 151, 158]],
 
        [[167, 183, 189],
         [167, 183, 189],
         [167, 183, 189],
         ...,
         [142, 154, 160],
         [141, 151, 158],
         [140, 150, 157]],
 
        ...,
 
        [[125, 175, 187],
         [123, 173, 185],
         [121, 170, 184],
         ...,
         [ 74, 115, 137],
         [ 74, 115, 137],
         [ 75, 116, 138]],
 
        [[119, 169, 181],
         [119, 169, 181],
         [120, 169, 183],
         ...,
   

##Visualização dos resultados

As imagens resultantes foram exibidas com as detecções realizadas pelo modelo, incluindo as bounding boxes, classes previstas e níveis de confiança.

Para facilitar a visualização no ambiente do Colab, as imagens foram redimensionadas durante a exibição.

In [29]:
for img in images[:5]:
    display(Image(filename=img, width=350))

Output hidden; open in https://colab.research.google.com to view.

## Conclusão do treinamento (30 épocas)

Após 30 épocas de treinamento, o modelo apresentou desempenho elevado, com alta precisão (~98%) e recall de 100%, indicando que todos os objetos foram corretamente identificados.

O valor de mAP50 próximo de 1 demonstra excelente capacidade de detecção, enquanto o mAP50-95 (~77%) indica que ainda há margem para melhorias na precisão das delimitações das caixas.

De forma geral, os resultados evidenciam que o modelo foi capaz de aprender de maneira eficaz mesmo com um conjunto de dados reduzido, demonstrando o potencial da abordagem de visão computacional utilizando YOLO.

## Treinamento do modelo (60 épocas)

O modelo foi treinado utilizando o dataset configurado no arquivo dataset.yaml, por um total de 60 épocas.

Foram mantidos os mesmos parâmetros do experimento anterior, como o tamanho das imagens (imgsz=640) e o tamanho do lote (batch=8), garantindo uma comparação justa entre os testes.

O aumento do número de épocas teve como objetivo avaliar se um treinamento mais longo poderia melhorar o desempenho do modelo na detecção dos objetos (mouse e apontador) e sua capacidade de generalização.

Esse experimento permite analisar o comportamento do modelo em relação ao tempo de treinamento, sem alterar outras variáveis do processo.

In [43]:
results_60 = model.train(
    data="/content/drive/MyDrive/Fase_6/dataset.yaml",
    epochs=60,
    imgsz=640,
    batch=8,
    name="exp_60_epochs"
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Fase_6/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/exp_30_epochs-2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp_60_epochs-3, nbs=64, nms=False, opset=None, 

## Limpeza da pasta de predições

Após o treinamento do modelo com 60 épocas, foi realizada a limpeza da pasta de predições (predict) antes da execução da etapa de inferência.

Esse procedimento foi necessário para evitar a mistura de resultados gerados em execuções anteriores, garantindo que apenas as predições mais recentes fossem consideradas na análise.

Dessa forma, assegura-se a organização correta dos resultados e a confiabilidade da avaliação visual do modelo.

In [49]:
import shutil
import os

predict_path = "/content/runs/detect/predict"

if os.path.exists(predict_path):
    shutil.rmtree(predict_path)
    print("Pasta predict limpa com sucesso ✔️")
else:
    print("Pasta predict não existe ainda")

Pasta predict limpa com sucesso ✔️


## Inferência com o modelo treinado

Após o treinamento, o modelo foi carregado a partir dos melhores pesos gerados (best.pt) e utilizado para realizar predições em imagens do conjunto de teste.

Essa etapa, conhecida como inferência, permite avaliar na prática a capacidade do modelo em identificar e localizar os objetos (mouse e apontador) em novas imagens não utilizadas no treinamento.

In [50]:
model = YOLO("/content/runs/detect/exp_60_epochs-3/weights/best.pt")

model.predict(
    source="/content/drive/MyDrive/Fase_6/images/test",
    save=True,
    conf=0.25
)


image 1/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_33.jpg: 640x480 1 apontador, 7.2ms
image 2/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_34.jpg: 640x480 1 apontador, 6.5ms
image 3/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_35.jpg: 640x480 1 apontador, 6.5ms
image 4/8 /content/drive/MyDrive/Fase_6/images/test/Apontador_36.jpg: 640x480 1 apontador, 6.4ms
image 5/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_33.jpg: 640x480 1 mouse, 20.6ms
image 6/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_34.jpg: 640x480 1 mouse, 6.3ms
image 7/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_35.jpg: 640x480 1 mouse, 6.3ms
image 8/8 /content/drive/MyDrive/Fase_6/images/test/Mouse_36.jpg: 640x480 1 mouse, 6.7ms
Speed: 3.5ms preprocess, 8.3ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/detect/predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'mouse', 1: 'apontador'}
 obb: None
 orig_img: array([[[167, 183, 189],
         [167, 183, 189],
         [169, 185, 191],
         ...,
         [138, 150, 156],
         [140, 150, 157],
         [141, 151, 158]],
 
        [[167, 183, 189],
         [167, 183, 189],
         [168, 184, 190],
         ...,
         [141, 153, 159],
         [141, 151, 158],
         [141, 151, 158]],
 
        [[167, 183, 189],
         [167, 183, 189],
         [167, 183, 189],
         ...,
         [142, 154, 160],
         [141, 151, 158],
         [140, 150, 157]],
 
        ...,
 
        [[125, 175, 187],
         [123, 173, 185],
         [121, 170, 184],
         ...,
         [ 74, 115, 137],
         [ 74, 115, 137],
         [ 75, 116, 138]],
 
        [[119, 169, 181],
         [119, 169, 181],
         [120, 169, 183],
         ...,
   

## Visualização dos resultados

As imagens resultantes foram exibidas com as detecções realizadas pelo modelo, incluindo as bounding boxes, classes previstas e níveis de confiança.

Para facilitar a visualização no ambiente do Colab, as imagens foram redimensionadas durante a exibição.

In [51]:
from IPython.display import Image, display
import glob

images = glob.glob('/content/runs/detect/predict/*.jpg')

for img in images[:5]:
    display(Image(filename=img, width=350))

Output hidden; open in https://colab.research.google.com to view.

## Avaliação do treinamento (60 épocas)

O modelo treinado por 60 épocas apresentou desempenho muito semelhante ao obtido no experimento com 30 épocas. As métricas de avaliação foram: Precision de 0.977, Recall de 1.000, mAP50 de 0.995 e mAP50-95 de 0.746.

Esses resultados indicam que o modelo já havia atingido um nível de convergência com menos épocas, não apresentando ganhos significativos com o aumento do treinamento.

Observa-se ainda uma leve redução no mAP50-95 em relação ao experimento anterior, sugerindo possível saturação do aprendizado ou início de overfitting leve.

## Comparação entre 30 e 60 épocas

A comparação entre os dois experimentos mostrou que o aumento do número de épocas de 30 para 60 não resultou em melhorias significativas no desempenho do modelo.

No treinamento com 30 épocas, foram obtidos os seguintes resultados: Precision de 0.981, Recall de 1.000, mAP50 de 0.995 e mAP50-95 de 0.775.

Já no treinamento com 60 épocas, as métricas foram: Precision de 0.977, Recall de 1.000, mAP50 de 0.995 e mAP50-95 de 0.746.

Ambos os modelos apresentaram desempenho muito próximo, com alta precisão e recall, indicando que o aprendizado já havia convergido no primeiro treinamento.

No entanto, o aumento das épocas não trouxe ganhos em generalização e resultou em leve queda na métrica mAP50-95, reforçando que o desempenho do modelo está mais relacionado à qualidade e diversidade do dataset do que ao tempo de treinamento.